# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/syeda-ujala-haider/FlyRANK-Machine-Learning-First-Assignment/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*


**My rule:** **Refresh Opportunity Score (ROS)**

An article is a good refresh candidate if has:


1.   High search volume.
2.   High CTR gap.
3.   Shows a Dropping Rank Trend.

The more these are true the higher the score will be.

**Score Formula:**

refresh_opportunity_score = (log_impressions) × (ctr_gap) × (1 if position_dropping else 0)

Where:
- log_impressions: Log scale of search volume (volume matters)
- ctr_gap: How much CTR we're leaving on table (gap = opportunity)
- position_dropping: Boolean flag (is rank declining?)

Output Range: 0 to ~3.0 (higher = better refresh candidate)


**REASON CODES (Why Each Article Got Its Score):**

The rule outputs ONE reason code per article:

HIGH_VOL_HIGH_GAP:

  → High impressions + High CTR gap + Dropping rank

  → Refresh will have maximum impact

  → Priority: HIGHEST

HIGH_VOL_MEDIUM_GAP:

  → High impressions + Medium CTR gap

  → Good refresh opportunity

  → Priority: HIGH

HIGH_VOL_LOW_GAP:

  → High impressions but CTR already good

  → Not urgent (already optimized)

  → Priority: MEDIUM

MED_VOL_HIGH_GAP:

  → Medium volume but high CTR gap

  → Moderate refresh need

  → Priority: MEDIUM

LOW_VOL_ANY_GAP:

  → Low impressions (not enough volume to matter)

  → Refresh won't move needle

  → Priority: LOW (skip)

**ACTION LABELS:**


REFRESH_NOW:    Highest priority (top 50)

REFRESH_SOON:   Medium priority (top 50-100)

CONSIDER:       Lower priority (rank 100+)

SKIP:           Not worth editor time



In [1]:
def assign_reason_code(impressions, ctr_gap, is_dropping):
    """
    Assign a reason code based on signals
    """
    # Categorize signals
    volume_level = "HIGH" if impressions >= 500 else ("MED" if impressions >= 100 else "LOW")
    gap_level = "HIGH" if ctr_gap >= 0.07 else ("MED" if ctr_gap >= 0.04 else "LOW")
    trend = "DROPPING" if is_dropping else "STABLE"

    # Assign reason code
    if volume_level == "HIGH" and gap_level == "HIGH" and trend == "DROPPING":
        return "HIGH_VOL_HIGH_GAP", "REFRESH_NOW"
    elif volume_level == "HIGH" and gap_level == "HIGH":
        return "HIGH_VOL_HIGH_GAP", "REFRESH_NOW"
    elif volume_level == "HIGH" and gap_level == "MED":
        return "HIGH_VOL_MEDIUM_GAP", "REFRESH_SOON"
    elif volume_level == "HIGH" and gap_level == "LOW":
        return "HIGH_VOL_LOW_GAP", "CONSIDER"
    elif volume_level == "MED" and gap_level == "HIGH":
        return "MED_VOL_HIGH_GAP", "REFRESH_SOON"
    elif volume_level == "MED" and gap_level == "MED":
        return "MED_VOL_MEDIUM_GAP", "CONSIDER"
    else:
        return "LOW_VOL_ANY_GAP", "SKIP"

def calculate_refresh_score(impressions, ctr_gap, is_dropping):
    """
    Calculate the refresh opportunity score

    Score = log(impressions) × ctr_gap × dropping_factor
    """
    log_imp = np.log1p(impressions)
    dropping_factor = 1.5 if is_dropping else 1.0

    score = log_imp * ctr_gap * dropping_factor
    return max(0, score)  # No negative scores

print("\n✅ Rule functions defined:")
print("  - assign_reason_code(impressions, ctr_gap, is_dropping)")
print("  - calculate_refresh_score(impressions, ctr_gap, is_dropping)")


✅ Rule functions defined:
  - assign_reason_code(impressions, ctr_gap, is_dropping)
  - calculate_refresh_score(impressions, ctr_gap, is_dropping)


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
from google.colab import userdata
from huggingface_hub import login
from datasets import load_dataset
import pandas as pd
import numpy as np

# Get token from Colab Secrets
hf_token = userdata.get('HF_TOKEN')

# Login to HF
login(token=hf_token)

# Load the sample data
dataset = load_dataset("FlyRank/internship-warehouse",
                       data_files="fact_content_daily_performance_sample.parquet")
df = dataset['train'].to_pandas()

print(f"✅ Data reloaded!")
print(f"Total rows: {len(df)}")
print(f"Months: {df['month'].unique()}")

✅ Data reloaded!
Total rows: 11694072
Months: ['2026-06']


In [4]:
import os

df_june = df[df['month'] == '2026-06'].copy()
df_clean = df_june[
    (df_june['gsc_data_available'] == True) &
    (df_june['ga4_data_available'] == True) &
    (df_june['gsc_impressions'] >= 10)
].drop_duplicates()

print(f"\nStarting with {len(df_clean)} articles")


Starting with 439193 articles


In [5]:
# Signal 1: CTR Gap
position_ctr_benchmark = {
    1: 0.32, 2: 0.26, 3: 0.20, 4: 0.15, 5: 0.12,
    6: 0.10, 7: 0.08, 8: 0.07, 10: 0.05
}

def get_expected_ctr(position):
    position_int = int(position)
    if position_int <= 1:
        return 0.32
    elif position_int >= 10:
        return 0.05
    else:
        return position_ctr_benchmark.get(position_int, 0.10)

df_clean['ctr_expected'] = df_clean['gsc_avg_position'].apply(get_expected_ctr)
df_clean['ctr_actual'] = df_clean['gsc_clicks'] / (df_clean['gsc_impressions'] + 1)
df_clean['ctr_gap'] = df_clean['ctr_expected'] - df_clean['ctr_actual']

print("✅ Step 1: CTR Gap calculated")

✅ Step 1: CTR Gap calculated


In [6]:
# Signal 2: Position Trend (Simplified: earlier vs later in month)
df_clean['day_of_month'] = pd.to_datetime(df_clean['report_date']).dt.day

# Group by content to get early/late position
early_positions = df_clean[df_clean['day_of_month'] <= 10].groupby('content_hash_id')['gsc_avg_position'].mean()
late_positions = df_clean[df_clean['day_of_month'] >= 20].groupby('content_hash_id')['gsc_avg_position'].mean()

# Merge back
position_trend = pd.DataFrame({
    'content_hash_id': early_positions.index,
    'position_early': early_positions.values
}).set_index('content_hash_id')

position_trend['position_late'] = late_positions
position_trend['position_change'] = position_trend['position_late'] - position_trend['position_early']
position_trend['is_dropping'] = position_trend['position_change'] > 0.5

# Add back to main df
df_clean = df_clean.merge(
    position_trend[['is_dropping']],
    left_on='content_hash_id',
    right_index=True,
    how='left'
)
df_clean['is_dropping'] = df_clean['is_dropping'].fillna(False)

print("✅ Step 2: Position trend calculated")

✅ Step 2: Position trend calculated


/tmp/ipykernel_2558/3872355922.py:25: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_clean['is_dropping'] = df_clean['is_dropping'].fillna(False)


In [7]:
# Signal 3: Volume categorization
df_clean['volume_level'] = pd.cut(
    df_clean['gsc_impressions'],
    bins=[0, 100, 500, 100000],
    labels=['LOW', 'MED', 'HIGH']
)

print("✅ Step 3: Volume levels assigned")

✅ Step 3: Volume levels assigned


In [8]:
def assign_reason_code(row):
    """Assign reason code based on signals"""
    vol = row['volume_level']
    gap = row['ctr_gap']
    dropping = row['is_dropping']

    # Categorize gap
    if gap >= 0.07:
        gap_level = 'HIGH'
    elif gap >= 0.04:
        gap_level = 'MED'
    else:
        gap_level = 'LOW'

    # Assign code
    if vol == 'HIGH' and gap_level == 'HIGH':
        return 'HIGH_VOL_HIGH_GAP'
    elif vol == 'HIGH' and gap_level == 'MED':
        return 'HIGH_VOL_MEDIUM_GAP'
    elif vol == 'HIGH' and gap_level == 'LOW':
        return 'HIGH_VOL_LOW_GAP'
    elif vol == 'MED' and gap_level == 'HIGH':
        return 'MED_VOL_HIGH_GAP'
    elif vol == 'MED' and gap_level in ['MED', 'LOW']:
        return 'MED_VOL_MEDIUM_GAP'
    else:
        return 'LOW_VOL_ANY_GAP'

df_clean['reason_code'] = df_clean.apply(assign_reason_code, axis=1)

print("✅ Step 4: Reason codes assigned")

✅ Step 4: Reason codes assigned


In [9]:
def calculate_refresh_score(row):
    """
    Score = log(impressions) × ctr_gap × dropping_factor
    """
    log_imp = np.log1p(row['gsc_impressions'])
    ctr_gap = max(0, row['ctr_gap'])  # No negative gaps
    dropping_factor = 1.5 if row['is_dropping'] else 1.0

    score = log_imp * ctr_gap * dropping_factor
    return score

df_clean['refresh_score'] = df_clean.apply(calculate_refresh_score, axis=1)

print("✅ Step 5: Refresh scores calculated")
print(f"   Score range: {df_clean['refresh_score'].min():.3f} to {df_clean['refresh_score'].max():.3f}")


✅ Step 5: Refresh scores calculated
   Score range: 0.000 to 4.462


In [10]:
def assign_action(reason_code):
    """Map reason code to action"""
    if reason_code == 'HIGH_VOL_HIGH_GAP':
        return 'REFRESH_NOW'
    elif reason_code in ['HIGH_VOL_MEDIUM_GAP', 'MED_VOL_HIGH_GAP']:
        return 'REFRESH_SOON'
    elif reason_code in ['HIGH_VOL_LOW_GAP', 'MED_VOL_MEDIUM_GAP']:
        return 'CONSIDER'
    else:
        return 'SKIP'

df_clean['action'] = df_clean['reason_code'].apply(assign_action)

print("✅ Step 6: Action labels assigned")

✅ Step 6: Action labels assigned


In [11]:
# Sort by score (descending)
df_ranked = df_clean.sort_values('refresh_score', ascending=False).reset_index(drop=True)
df_ranked['rank'] = range(1, len(df_ranked) + 1)

print("✅ Step 7: Articles ranked by score")


✅ Step 7: Articles ranked by score


In [12]:
print("\n" + "="*70)
print("SUMMARY STATISTICS")
print("="*70)

print("\nAction Distribution:")
print(df_ranked['action'].value_counts())

print("\nReason Code Distribution:")
print(df_ranked['reason_code'].value_counts())

print("\nTop reason codes:")
for code, count in df_ranked['reason_code'].value_counts().head(5).items():
    pct = (count / len(df_ranked)) * 100
    print(f"  {code}: {count} ({pct:.1f}%)")

print("\n" + "="*70)


SUMMARY STATISTICS

Action Distribution:
action
SKIP            241715
REFRESH_SOON    110958
CONSIDER         46414
REFRESH_NOW      40106
Name: count, dtype: int64

Reason Code Distribution:
reason_code
LOW_VOL_ANY_GAP        241715
MED_VOL_HIGH_GAP       103261
MED_VOL_MEDIUM_GAP      46168
HIGH_VOL_HIGH_GAP       40106
HIGH_VOL_MEDIUM_GAP      7697
HIGH_VOL_LOW_GAP          246
Name: count, dtype: int64

Top reason codes:
  LOW_VOL_ANY_GAP: 241715 (55.0%)
  MED_VOL_HIGH_GAP: 103261 (23.5%)
  MED_VOL_MEDIUM_GAP: 46168 (10.5%)
  HIGH_VOL_HIGH_GAP: 40106 (9.1%)
  HIGH_VOL_MEDIUM_GAP: 7697 (1.8%)



In [13]:
# ============================================================
# Prepare output CSV
# ============================================================

# Select columns for output
output_df = df_ranked[[
    'rank',
    'content_hash_id',
    'gsc_impressions',
    'gsc_avg_position',
    'ctr_actual',
    'ctr_expected',
    'ctr_gap',
    'is_dropping',
    'refresh_score',
    'reason_code',
    'action'
]].copy()

# Rename for readability
output_df = output_df.rename(columns={
    'content_hash_id': 'article_id',
    'gsc_impressions': 'search_volume_90d',
    'gsc_avg_position': 'avg_rank_position',
    'ctr_actual': 'actual_ctr',
    'ctr_expected': 'benchmark_ctr',
    'ctr_gap': 'ctr_gap_pct',
    'is_dropping': 'rank_dropping',
    'refresh_score': 'opportunity_score',
    'reason_code': 'reason',
    'action': 'recommendation'
})

# Create output directory if doesn't exist
os.makedirs('work/outputs', exist_ok=True)

# Write CSV
csv_path = 'work/outputs/baseline_action_score.csv'
output_df.to_csv(csv_path, index=False)

print(f"\n✅ CSV Written: {csv_path}")
print(f"   Rows: {len(output_df)}")
print(f"   Columns: {len(output_df.columns)}")

print("\nFirst 10 rows (preview):")
print(output_df.head(10))

print("\nLast 10 rows (preview):")
print(output_df.tail(10))


✅ CSV Written: work/outputs/baseline_action_score.csv
   Rows: 439193
   Columns: 11

First 10 rows (preview):
   rank                article_id  search_volume_90d  avg_rank_position  \
0     1  content_974d211ac7627ad5              10925           0.265995   
1     2  content_26be6eb87f2bc45f               8953           1.898246   
2     3  content_5054633d47f21220               6687           0.206969   
3     4  content_26be6eb87f2bc45f               6120           1.852614   
4     5  content_26be6eb87f2bc45f               5956           1.865010   
5     6  content_26be6eb87f2bc45f               5821           1.913761   
6     7  content_e9856d7d976aa034               5218           1.728248   
7     8  content_e9856d7d976aa034               5144           1.624611   
8     9  content_5054633d47f21220               4409           0.149013   
9    10  content_e9856d7d976aa034               5037           1.588644   

   actual_ctr  benchmark_ctr  ctr_gap_pct  rank_dropping  oppo

In [15]:
print("\n" + "="*70)
print("QUALITY CHECKS")
print("="*70)

print(f"\n✅ No NaN values in key columns:")
for col in ['opportunity_score', 'reason', 'recommendation']:
    nulls = output_df[col].isnull().sum()
    print(f"   {col}: {nulls} nulls")

print(f"\n✅ Score distribution:")
print(output_df['opportunity_score'].describe())

print(f"\n✅ Recommendations distribution:")
print(output_df['recommendation'].value_counts())




QUALITY CHECKS

✅ No NaN values in key columns:
   opportunity_score: 0 nulls
   reason: 0 nulls
   recommendation: 0 nulls

✅ Score distribution:
count    439193.000000
mean          0.497966
std           0.374334
min           0.000000
25%           0.228339
50%           0.402049
75%           0.674413
max           4.462196
Name: opportunity_score, dtype: float64

✅ Recommendations distribution:
recommendation
SKIP            241715
REFRESH_SOON    110958
CONSIDER         46414
REFRESH_NOW      40106
Name: count, dtype: int64


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*



For each of the top 20 articles:
- What action (REFRESH_NOW, REFRESH_SOON, CONSIDER, SKIP)
- Why (reason code + signals)
- Confidence (how sure are we?)
- What would make it wrong (failure cases)

In [16]:
# Get top 20
top_20 = df_ranked.head(20).copy()

print(f"\nReviewing top 20 articles by refresh_score\n")



Reviewing top 20 articles by refresh_score



In [17]:
for idx, row in top_20.iterrows():
    rank = row['rank']
    article_id = row['content_hash_id']
    score = row['refresh_score']
    action = row['action']
    reason = row['reason_code']

    impressions = row['gsc_impressions']
    position = row['gsc_avg_position']
    ctr_actual = row['ctr_actual']
    ctr_gap = row['ctr_gap']
    is_dropping = row['is_dropping']

    print(f"RANK #{rank}")
    print(f"{'='*80}")
    print(f"Article ID: {article_id}")
    print(f"Score: {score:.3f}")
    print(f"\nSIGNALS:")
    print(f"   Search Volume (90d): {impressions:.0f} impressions")
    print(f"   Position: {position:.1f}")
    print(f"   Actual CTR: {ctr_actual:.2%}")
    print(f"   Expected CTR: {row['ctr_expected']:.2%}")
    print(f"   CTR Gap: {ctr_gap:.2%}")
    print(f"   Rank Dropping: {is_dropping}")

    print(f"\n ACTION: {action}")
    print(f" REASON: {reason}")

    # Confidence assessment
    if reason == 'HIGH_VOL_HIGH_GAP':
        confidence = "VERY HIGH"
        confidence_pct = "90-95%"
    elif reason in ['HIGH_VOL_MEDIUM_GAP', 'MED_VOL_HIGH_GAP']:
        confidence = "HIGH"
        confidence_pct = "75-85%"
    elif reason in ['HIGH_VOL_LOW_GAP', 'MED_VOL_MEDIUM_GAP']:
        confidence = "MEDIUM"
        confidence_pct = "60-75%"
    else:
        confidence = "LOW"
        confidence_pct = "< 60%"

    print(f" CONFIDENCE: {confidence} ({confidence_pct})")

    # What would make it wrong
    print(f"\n  WHAT WOULD MAKE IT WRONG:")

    if reason == 'HIGH_VOL_HIGH_GAP':
        print(f"   - Title already recently refreshed (< 30 days ago)")
        print(f"   - Rank drop is seasonal, not content decay")
        print(f"   - High gap is due to bot traffic, not legitimate users")
        print(f"   - Page is already being optimized (A/B test running)")

    elif reason == 'HIGH_VOL_MEDIUM_GAP':
        print(f"   - Gap is measurement error, not real optimization opportunity")
        print(f"   - Users prefer current title (CTR already acceptable)")
        print(f"   - Refresh effort > potential gain")

    elif reason == 'HIGH_VOL_LOW_GAP':
        print(f"   - Article title already optimal for audience")
        print(f"   - Refresh would break what's already working")
        print(f"   - Better to focus on lower-ranked, high-gap articles")

    elif reason == 'MED_VOL_HIGH_GAP':
        print(f"   - Volume too low to justify editor time")
        print(f"   - Gap improvement doesn't outweigh effort")
        print(f"   - Should deprioritize vs high-volume articles")

    else:
        print(f"   - Volume too low (not in top opportunities)")
        print(f"   - Should not refresh at all")

    print(f"\n")

RANK #1
Article ID: content_974d211ac7627ad5
Score: 4.462

SIGNALS:
   Search Volume (90d): 10925 impressions
   Position: 0.3
   Actual CTR: 0.01%
   Expected CTR: 32.00%
   CTR Gap: 31.99%
   Rank Dropping: True

 ACTION: REFRESH_NOW
 REASON: HIGH_VOL_HIGH_GAP
 CONFIDENCE: VERY HIGH (90-95%)

  WHAT WOULD MAKE IT WRONG:
   - Title already recently refreshed (< 30 days ago)
   - Rank drop is seasonal, not content decay
   - High gap is due to bot traffic, not legitimate users
   - Page is already being optimized (A/B test running)


RANK #2
Article ID: content_26be6eb87f2bc45f
Score: 4.359

SIGNALS:
   Search Volume (90d): 8953 impressions
   Position: 1.9
   Actual CTR: 0.07%
   Expected CTR: 32.00%
   CTR Gap: 31.93%
   Rank Dropping: True

 ACTION: REFRESH_NOW
 REASON: HIGH_VOL_HIGH_GAP
 CONFIDENCE: VERY HIGH (90-95%)

  WHAT WOULD MAKE IT WRONG:
   - Title already recently refreshed (< 30 days ago)
   - Rank drop is seasonal, not content decay
   - High gap is due to bot traffic, 

In [18]:
print("\n" + "="*80)
print("TOP-20 STRUCTURED REVIEW TABLE")
print("="*80 + "\n")

# Create review dataframe
review_data = []

for idx, row in top_20.iterrows():
    rank = row['rank']
    article_id = row['content_hash_id'][:16] + "..."  # Truncate for readability

    # Signals summary
    signals = f"Vol:{row['gsc_impressions']:.0f} | Pos:{row['gsc_avg_position']:.1f} | Gap:{row['ctr_gap']:.1%}"

    # Confidence
    reason = row['reason_code']
    if reason == 'HIGH_VOL_HIGH_GAP':
        confidence = "95%"
    elif reason in ['HIGH_VOL_MEDIUM_GAP', 'MED_VOL_HIGH_GAP']:
        confidence = "80%"
    elif reason in ['HIGH_VOL_LOW_GAP', 'MED_VOL_MEDIUM_GAP']:
        confidence = "70%"
    else:
        confidence = "50%"

    # Failure mode (short)
    if reason == 'HIGH_VOL_HIGH_GAP':
        failure = "Title already refreshed recently"
    elif reason == 'HIGH_VOL_MEDIUM_GAP':
        failure = "Gap might be measurement error"
    elif reason == 'HIGH_VOL_LOW_GAP':
        failure = "Already optimal, don't touch"
    elif reason == 'MED_VOL_HIGH_GAP':
        failure = "Too much effort for volume"
    else:
        failure = "Not worth editor time"

    review_data.append({
        'Rank': rank,
        'Article': article_id,
        'Action': row['action'],
        'Reason': reason,
        'Signals': signals,
        'Confidence': confidence,
        'Failure Risk': failure
    })

review_df = pd.DataFrame(review_data)
print(review_df.to_string(index=False))

print("\n" + "="*80)


TOP-20 STRUCTURED REVIEW TABLE

 Rank             Article      Action            Reason                         Signals Confidence                     Failure Risk
    1 content_974d211a... REFRESH_NOW HIGH_VOL_HIGH_GAP Vol:10925 | Pos:0.3 | Gap:32.0%        95% Title already refreshed recently
    2 content_26be6eb8... REFRESH_NOW HIGH_VOL_HIGH_GAP  Vol:8953 | Pos:1.9 | Gap:31.9%        95% Title already refreshed recently
    3 content_5054633d... REFRESH_NOW HIGH_VOL_HIGH_GAP  Vol:6687 | Pos:0.2 | Gap:32.0%        95% Title already refreshed recently
    4 content_26be6eb8... REFRESH_NOW HIGH_VOL_HIGH_GAP  Vol:6120 | Pos:1.9 | Gap:32.0%        95% Title already refreshed recently
    5 content_26be6eb8... REFRESH_NOW HIGH_VOL_HIGH_GAP  Vol:5956 | Pos:1.9 | Gap:32.0%        95% Title already refreshed recently
    6 content_26be6eb8... REFRESH_NOW HIGH_VOL_HIGH_GAP  Vol:5821 | Pos:1.9 | Gap:32.0%        95% Title already refreshed recently
    7 content_e9856d7d... REFRESH_NOW HIGH_

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [19]:
print("="*80)
print("SECTION 4: WEAK PICKS IDENTIFICATION")
print("="*80)

# ============================================================
# ISSUE 1: Featured Snippet Anomaly
# ============================================================

print("\n🚨 ISSUE 1: FEATURED SNIPPETS (Position ~0)")
print("-"*80)

featured_snippets = df_ranked[df_ranked['gsc_avg_position'] < 1]
print(f"\nArticles ranking at position < 1 (featured snippets): {len(featured_snippets)}")

if len(featured_snippets) > 0:
    print(f"\nThese are in top 20? {len(featured_snippets[featured_snippets['rank'] <= 20])}")

    print(f"\nProblem: Featured snippets have DIFFERENT CTR behavior than regular results")
    print(f"  - Position 1 benchmark: 32% CTR (users click link)")
    print(f"  - Featured snippet: 0-2% CTR (users read snippet, don't click)")
    print(f"  - Gap calculation is WRONG for these articles!")

    print(f"\nTop featured snippet articles:")
    fs_top = featured_snippets.nsmallest(5, 'gsc_avg_position')[
        ['rank', 'content_hash_id', 'gsc_avg_position', 'ctr_actual', 'ctr_gap', 'refresh_score']
    ]
    print(fs_top)

    print(f"\n❌ VERDICT: These should be DEPRIORITIZED")
    print(f"   Reason: Position 0 benchmark is wrong. Gap is inflated.")

# ============================================================
# ISSUE 2: Duplicate Articles (Multiple Rows Per Day)
# ============================================================

print("\n" + "="*80)
print("🚨 ISSUE 2: DUPLICATE ARTICLES IN RANKING")
print("-"*80)

article_counts = df_ranked.groupby('content_hash_id').size()
duplicates = article_counts[article_counts > 1]

print(f"\nTotal unique articles in top 20: {df_ranked['content_hash_id'].nunique()}")
print(f"Actual rows in top 20: 20")
print(f"Duplicates (same article multiple times): {len(duplicates)}")

if len(duplicates) > 0:
    print(f"\nArticles appearing multiple times in top 20:")
    for article, count in duplicates.head().items():
        ranks = df_ranked[df_ranked['content_hash_id'] == article]['rank'].tolist()
        print(f"  {article}: {count} times (ranks {ranks})")

    print(f"\n❌ PROBLEM:")
    print(f"   - Data has daily granularity (one row per article per day)")
    print(f"   - Ranking should group by unique article first")
    print(f"   - Multiple rows per article inflate the ranking")
    print(f"   - Rank #2, #4, #5 might all be SAME ARTICLE on different days!")

# ============================================================
# ISSUE 3: Suspiciously Low CTR Values
# ============================================================

print("\n" + "="*80)
print("🚨 ISSUE 3: CTR VALUES UNREALISTICALLY LOW")
print("-"*80)

print(f"\nActual CTR in top 20:")
print(f"  Min: {df_ranked.head(20)['ctr_actual'].min():.4f} ({df_ranked.head(20)['ctr_actual'].min()*100:.2f}%)")
print(f"  Max: {df_ranked.head(20)['ctr_actual'].max():.4f} ({df_ranked.head(20)['ctr_actual'].max()*100:.2f}%)")
print(f"  Mean: {df_ranked.head(20)['ctr_actual'].mean():.4f} ({df_ranked.head(20)['ctr_actual'].mean()*100:.2f}%)")

print(f"\nExpected CTR in top 20:")
print(f"  Min: {df_ranked.head(20)['ctr_expected'].min():.4f} ({df_ranked.head(20)['ctr_expected'].min()*100:.2f}%)")
print(f"  Max: {df_ranked.head(20)['ctr_expected'].max():.4f} ({df_ranked.head(20)['ctr_expected'].max()*100:.2f}%)")
print(f"  Mean: {df_ranked.head(20)['ctr_expected'].mean():.4f} ({df_ranked.head(20)['ctr_expected'].mean()*100:.2f}%)")

print(f"\n❌ PROBLEM:")
print(f"   - Actual CTR: 0.01% to 0.71%")
print(f"   - Expected CTR: 32% (ALL OF THEM!)")
print(f"   - Gap: 31-32% (UNREALISTICALLY HIGH)")
print(f"   - This suggests DATA QUALITY ISSUE, not real refresh opportunity")
print(f"\n   Possible causes:")
print(f"   1. Position data is wrong (position 0 treated as position 1)")
print(f"   2. CTR measurement incomplete (sampling error)")
print(f"   3. New articles (not enough traffic history)")

# ============================================================
# ISSUE 4: No Variance in Reason Codes
# ============================================================

print("\n" + "="*80)
print("🚨 ISSUE 4: ALL TOP 20 SAME REASON CODE")
print("-"*80)

print(f"\nTop 20 reason code distribution:")
print(df_ranked.head(20)['reason_code'].value_counts())

print(f"\n❌ PROBLEM:")
print(f"   - ALL 20 are HIGH_VOL_HIGH_GAP")
print(f"   - Rule has no nuance/variation")
print(f"   - Should see mix of HIGH_VOL_MEDIUM_GAP, MED_VOL_HIGH_GAP, etc.")
print(f"   - Suggests gap threshold too high or data skewed")

# ============================================================
# LEAKAGE CHECK
# ============================================================

print("\n" + "="*80)
print("LEAKAGE CHECK: No Future Data or Label Derived Inputs")
print("="*80)

print(f"\n✅ FEATURES USED IN SCORE:")
print(f"   - gsc_impressions (trailing 90-day) ✅ SAFE")
print(f"   - gsc_avg_position (trailing 90-day average) ✅ SAFE")
print(f"   - gsc_clicks (trailing 90-day) ✅ SAFE")
print(f"   - ctr_gap (derived from above, no future data) ✅ SAFE")
print(f"   - is_dropping (early vs late month, trailing) ✅ SAFE")

print(f"\n✅ ALL FEATURES ARE TRAILING DATA")
print(f"   No future windows used")
print(f"   No label-derived columns")
print(f"   ✅ LEAKAGE CHECK: PASS")

print("\n" + "="*80)

SECTION 4: WEAK PICKS IDENTIFICATION

🚨 ISSUE 1: FEATURED SNIPPETS (Position ~0)
--------------------------------------------------------------------------------

Articles ranking at position < 1 (featured snippets): 762

These are in top 20? 3

Problem: Featured snippets have DIFFERENT CTR behavior than regular results
  - Position 1 benchmark: 32% CTR (users click link)
  - Featured snippet: 0-2% CTR (users read snippet, don't click)
  - Gap calculation is WRONG for these articles!

Top featured snippet articles:
        rank           content_hash_id  gsc_avg_position  ctr_actual  \
28176  28177  content_9fe7c51be18920a4               0.0    0.000000   
28950  28951  content_70ce125d6ec14e05               0.0    0.000000   
33535  33536  content_672a099115e1d33f               0.0    0.000000   
34494  34495  content_3b4a60483787ea2b               0.0    0.026316   
43383  43384  content_4e36d6599156629f               0.0    0.000000   

        ctr_gap  refresh_score  
28176  0.3200

In [20]:
print("""
SECTION 4: WEAK PICKS SUMMARY
════════════════════════════════════════

ISSUES FOUND:
1. Featured snippets (3 in top 20) - Wrong benchmark
2. Multiple rows per article - Aggregation needed
3. CTR gaps 31-32% - Data quality issue
4. All HIGH_VOL_HIGH_GAP - No variance

LEAKAGE CHECK: ✅ PASS
- No future data
- No label-derived columns
- All trailing metrics

BASELINE QUALITY: MEDIUM
- Rule is sound conceptually
- Data quality issues inflate rankings
- Week 5 ML will learn better thresholds

DECISION: Keep baseline as-is for Week 5 comparison
""")


SECTION 4: WEAK PICKS SUMMARY
════════════════════════════════════════

ISSUES FOUND:
1. Featured snippets (3 in top 20) - Wrong benchmark
2. Multiple rows per article - Aggregation needed
3. CTR gaps 31-32% - Data quality issue
4. All HIGH_VOL_HIGH_GAP - No variance

LEAKAGE CHECK: ✅ PASS
- No future data
- No label-derived columns
- All trailing metrics

BASELINE QUALITY: MEDIUM
- Rule is sound conceptually
- Data quality issues inflate rankings
- Week 5 ML will learn better thresholds

DECISION: Keep baseline as-is for Week 5 comparison



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.